# Time Value of Money

The time value of money is the foundation underneath discounting, bond pricing, discounted cash flow valuation, and portfolio return math. This notebook builds the core intuition and turns it into reusable Python functions.

Abbreviations used in this notebook:

- **TVM**: Time Value of Money, the idea that money today is worth more than the same amount in the future.
- **PV**: Present Value, what a future cash flow is worth today.
- **FV**: Future Value, what today's money can grow into later.
- **PMT**: Payment, a recurring cash flow in an annuity.
- **DCF**: Discounted Cash Flow, a valuation method based on present values of future cash flows.
- **CHF**: Swiss franc, the currency used in the examples.

## 1. Intuition

Money today is worth more than the same amount of money in the future because today’s money can be invested, inflation can reduce purchasing power, and future cash flows are uncertain.

There are two core operations:

- **Compounding** moves money forward in time.
- **Discounting** moves money backward to today.

A useful mental model: compounding asks, “What will this become?” Discounting asks, “What is that future amount worth today?”

## 2. Mathematics

Future value with annual compounding:

$$
FV = PV \times (1 + r)^n
$$

Present value:

$$
PV = \frac{FV}{(1 + r)^n}
$$

Future value with multiple compounding periods per year:

$$
FV = PV \times \left(1 + \frac{r}{m}\right)^{m \times n}
$$

Future value with continuous compounding:

$$
FV = PV \times e^{r \times n}
$$

Present value of an ordinary annuity:

$$
PV_{annuity} = PMT \times \frac{1 - (1 + r)^{-n}}{r}
$$

Where:

- $PV$ = present value, or the amount today.
- $FV$ = future value, or what the amount grows into.
- $PV_{annuity}$ = present value of a repeated payment stream.
- $PMT$ = recurring payment amount.
- $r$ = annual interest rate, return rate, or discount rate.
- $n$ = number of years.
- $m$ = number of compounding periods per year.
- $e$ = mathematical constant used for continuous compounding.

## 3. Implementation

We will create small reusable functions, then apply them to lump sums and recurring cash flows. The examples use CHF, but the formulas work for any currency.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.4f}".format


def future_value(present_value, rate, years):
    return present_value * (1 + rate) ** years


def present_value(future_value_amount, rate, years):
    return future_value_amount / (1 + rate) ** years


def future_value_periodic(present_value, rate, years, periods_per_year):
    return present_value * (1 + rate / periods_per_year) ** (periods_per_year * years)


def future_value_continuous(present_value, rate, years):
    return present_value * np.exp(rate * years)


def present_value_annuity(payment, rate, years):
    return payment * (1 - (1 + rate) ** (-years)) / rate

A simple round trip check confirms that compounding and discounting are inverse operations.

In [ ]:
pv = 1_000
rate = 0.05
years = 10

fv = future_value(pv, rate, years)
pv_back = present_value(fv, rate, years)

round_trip = pd.DataFrame({
    "metric": ["Present value", "Future value", "Discounted back to present"],
    "amount": [pv, fv, pv_back],
})

assert np.isclose(pv, pv_back)
round_trip.style.format({"amount": "CHF {:,.2f}"})

In [ ]:
scenarios = pd.DataFrame({
    "rate": [0.02, 0.05, 0.08],
    "years": [5, 10, 20],
})

rows = []
for _, scenario in scenarios.iterrows():
    rows.append({
        "rate": scenario["rate"],
        "years": int(scenario["years"]),
        "future_value_of_1000": future_value(1_000, scenario["rate"], scenario["years"]),
        "present_value_of_1000_due_later": present_value(1_000, scenario["rate"], scenario["years"]),
    })

pd.DataFrame(rows).style.format({
    "rate": "{:.1%}",
    "years": "{:,.0f}",
    "future_value_of_1000": "CHF {:,.2f}",
    "present_value_of_1000_due_later": "CHF {:,.2f}",
})

## 4. Visualization

The effect of time and rate is nonlinear. Small rate differences can become large over long horizons.

In [ ]:
years_range = np.arange(0, 31)
rates = [0.02, 0.05, 0.08]

fig, ax = plt.subplots(figsize=(9, 4.5))

for r in rates:
    ax.plot(years_range, future_value(1_000, r, years_range), marker="o", markersize=3, label=f"{r:.0%} return")

ax.set_title("Compounding CHF 1,000 Over Time")
ax.set_xlabel("Years")
ax.set_ylabel("Future value")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
discount_rates = np.linspace(0.01, 0.10, 10)
time_horizons = np.arange(1, 21)

pv_matrix = pd.DataFrame(index=discount_rates, columns=time_horizons, dtype=float)
for r in discount_rates:
    for n in time_horizons:
        pv_matrix.loc[r, n] = present_value(1_000, r, n)

fig, ax = plt.subplots(figsize=(10, 5))
image = ax.imshow(pv_matrix.values, aspect="auto", cmap="magma_r")

ax.set_title("Present Value of CHF 1,000 Received in the Future")
ax.set_xlabel("Years from today")
ax.set_ylabel("Discount rate")
ax.set_xticks(range(len(time_horizons)))
ax.set_xticklabels(time_horizons)
ax.set_yticks(range(len(discount_rates)))
ax.set_yticklabels([f"{r:.0%}" for r in discount_rates])

fig.colorbar(image, ax=ax, label="Present value")
plt.tight_layout()
plt.show()

## 5. Application

Time value of money appears almost everywhere in finance:

- **DCF valuation**: future free cash flows are discounted to present value.
- **Bond pricing**: coupons and principal repayment are discounted to today.
- **Loan amortization**: payments are split between interest and principal over time.
- **Retirement planning**: recurring savings compound over decades.

For company valuation, the concept matters because a franc of free cash flow today is more valuable than a franc of free cash flow ten years from now.

In [ ]:
cash_flows = pd.DataFrame({
    "year": [1, 2, 3, 4, 5],
    "cash_flow": [100, 110, 120, 130, 140],
})

cash_flows["present_value_at_6pct"] = cash_flows["cash_flow"] / (1 + 0.06) ** cash_flows["year"]

annuity_value = present_value_annuity(payment=100, rate=0.06, years=5)

print(f"PV of uneven cash flows: CHF {cash_flows['present_value_at_6pct'].sum():,.2f}")
print(f"PV of CHF 100 annuity for 5 years: CHF {annuity_value:,.2f}")

cash_flows.style.format({
    "year": "{:,.0f}",
    "cash_flow": "CHF {:,.2f}",
    "present_value_at_6pct": "CHF {:,.2f}",
})

## 6. Reflection

- Compounding is exponential, not linear.
- Discounting is the inverse of compounding.
- Time and interest rates interact: long horizons magnify small rate changes.
- Present value is the bridge between future cash flows and today’s investment decisions.
- The discount rate is not just math; it expresses opportunity cost, inflation, and risk.

Questions to answer after running the notebook:

1. Why does the same discount rate matter more over 20 years than over 2 years?
2. What happens to present value when risk increases?
3. Why is time value of money the first building block for DCF valuation?
4. When would continuous compounding matter in practice?